<a href="https://colab.research.google.com/github/MuhammadTalhaIqbal/linear-Regression-using-numpy-/blob/main/Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

### Load DenseNet201 with ImageNet weights (excluding the top classifier)

We will use `include_top=False` to exclude the final classification layer, and `weights='imagenet'` to load the pre-trained weights. The backbone will be frozen by setting `trainable=False`.

In [3]:
# Load DenseNet201 with pre-trained ImageNet weights, excluding the top classifier
base_model = DenseNet201(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the layers of the base model
for layer in base_model.layers:
    layer.trainable = False

print("DenseNet201 base model loaded and layers frozen.")

74836368/74836368 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
DenseNet201 base model loaded and layers frozen.


### Build a new classifier head

Now, we'll add our own classification layers on top of the frozen DenseNet201 backbone. This head will be responsible for learning to classify the new dataset.

In [4]:
# Create a new classifier head
x = base_model.output
x = GlobalAveragePooling2D()(x) # Add a Global Average Pooling layer
x = Dense(1024, activation='relu')(x) # Add a Dense layer with ReLU activation
predictions = Dense(10, activation='softmax')(x) # Output layer for 10 classes (example, adjust as needed)

# Combine the base model and the new head into a full model
model = Model(inputs=base_model.input, outputs=predictions)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 20,299,338 (77.44 MB)

 Trainable params: 1,977,354 (7.54 MB)

 Non-trainable params: 18,321,984 (69.89 MB)

### Compile the model

Finally, we compile the model, specifying the optimizer, loss function, and metrics. For a multi-class classification problem, `categorical_crossentropy` is a common choice, and `Adam` is a good general-purpose optimizer.

In [5]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("Model compiled successfully.")
print("Next, you will need to prepare your dataset for training.")

Model compiled successfully.
Next, you will need to prepare your dataset for training.


### Prepare Dummy Data for Demonstration

Since no specific dataset was provided, I'll create some dummy data for demonstration purposes. This will allow us to run the training process and observe the metrics.

In [6]:
import numpy as np

# Generate dummy data for demonstration
num_samples = 100
num_classes = 10 # This should match the output layer of your model

# Dummy images (batch_size, height, width, channels)
X_train_dummy = np.random.rand(num_samples, 224, 224, 3).astype('float32')

# Dummy labels (one-hot encoded for categorical_crossentropy)
y_train_dummy = tf.keras.utils.to_categorical(np.random.randint(0, num_classes, num_samples), num_classes=num_classes)

print(f"Dummy training data shape: {X_train_dummy.shape}")
print(f"Dummy training labels shape: {y_train_dummy.shape}")

Dummy training data shape: (100, 224, 224, 3)
Dummy training labels shape: (100, 10)


### Train the Model

Now, let's train the model using the dummy data. We'll capture the training history to extract accuracy, loss, and training time.

In [7]:
import time

epochs = 5 # For demonstration, you might want more epochs for real training
batch_size = 32

start_time = time.time()
history = model.fit(
    X_train_dummy,
    y_train_dummy,
    epochs=epochs,
    batch_size=batch_size,
    verbose=1 # Show training progress
)
end_time = time.time()

training_time = end_time - start_time

print(f"\nTraining complete in {training_time:.2f} seconds.")

Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 75s 8s/step - accuracy: 0.1200 - loss: 2.5943
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.1500 - loss: 2.7333
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.1500 - loss: 2.4862 
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - accuracy: 0.1500 - loss: 2.3042 
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.1300 - loss: 2.2943

Training complete in 77.02 seconds.


### Report Metrics: Accuracy, Loss, and Training Time

Here are the requested metrics from the training process:

In [8]:
# Extract metrics from history
final_accuracy = history.history['accuracy'][-1]
final_loss = history.history['loss'][-1]

print(f"Final Training Accuracy: {final_accuracy:.4f}")
print(f"Final Training Loss: {final_loss:.4f}")
print(f"Total Training Time: {training_time:.2f} seconds")

Final Training Accuracy: 0.1300
Final Training Loss: 2.2943
Total Training Time: 77.02 seconds


---
### Implementing ResNet50 with Frozen Backbone

Now, let's implement ResNet50 using the same approach: ImageNet pre-trained weights, freezing the backbone, and training only the classifier head. We'll use the same dummy data for consistency.

In [9]:
from tensorflow.keras.applications import ResNet50

# Load ResNet50 with pre-trained ImageNet weights, excluding the top classifier
base_model_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the layers of the base model
for layer in base_model_resnet.layers:
    layer.trainable = False

print("ResNet50 base model loaded and layers frozen.")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
ResNet50 base model loaded and layers frozen.


### Build a new classifier head for ResNet50

Similar to DenseNet201, we'll add our own classification layers on top of the frozen ResNet50 backbone.

In [10]:
# Create a new classifier head for ResNet50
x_resnet = base_model_resnet.output
x_resnet = GlobalAveragePooling2D()(x_resnet)
x_resnet = Dense(1024, activation='relu')(x_resnet)
predictions_resnet = Dense(num_classes, activation='softmax')(x_resnet) # Use num_classes from previous dummy data setup

# Combine the base model and the new head into a full model
model_resnet = Model(inputs=base_model_resnet.input, outputs=predictions_resnet)

model_resnet.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 25,696,138 (98.02 MB)

 Trainable params: 2,108,426 (8.04 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

### Compile the ResNet50 model

In [11]:
# Compile the ResNet50 model
model_resnet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("ResNet50 model compiled successfully.")

ResNet50 model compiled successfully.


### Train the ResNet50 Model

Training the ResNet50 model using the same dummy data and capturing metrics.

In [12]:
import time

# Reuse epochs and batch_size from previous setup

start_time_resnet = time.time()
history_resnet = model_resnet.fit(
    X_train_dummy, # Use the previously generated dummy data
    y_train_dummy, # Use the previously generated dummy labels
    epochs=epochs,
    batch_size=batch_size,
    verbose=1
)
end_time_resnet = time.time()

training_time_resnet = end_time_resnet - start_time_resnet

print(f"\nResNet50 Training complete in {training_time_resnet:.2f} seconds.")

Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.0600 - loss: 2.8135
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.1100 - loss: 2.6526 
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.1000 - loss: 2.5475 
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.0600 - loss: 2.5468 
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.0400 - loss: 2.5759

ResNet50 Training complete in 18.69 seconds.


### Report Metrics for ResNet50: Accuracy, Loss, and Training Time

In [13]:
# Extract metrics from history_resnet
final_accuracy_resnet = history_resnet.history['accuracy'][-1]
final_loss_resnet = history_resnet.history['loss'][-1]

print(f"ResNet50 Final Training Accuracy: {final_accuracy_resnet:.4f}")
print(f"ResNet50 Final Training Loss: {final_loss_resnet:.4f}")
print(f"ResNet50 Total Training Time: {training_time_resnet:.2f} seconds")

ResNet50 Final Training Accuracy: 0.0400
ResNet50 Final Training Loss: 2.5759
ResNet50 Total Training Time: 18.69 seconds


---
### Performance Comparison: DenseNet201, ResNet50, and ResNet101

Based on the training on the dummy dataset with 5 epochs, here's a side-by-side comparison of DenseNet201, ResNet50, and ResNet101:

| Metric              | DenseNet201       | ResNet50          | ResNet101         |
| :------------------ | :---------------- | :---------------- | :---------------- |
| Final Accuracy      | 0.1300            | 0.0400            | 0.1200            |
| Final Loss          | 2.2943            | 2.5759            | 2.3983            |
| Total Training Time | 77.02 seconds     | 18.69 seconds     | 28.72 seconds     |

**Observations:**

*   DenseNet201 achieved the highest final accuracy and lowest final loss among the three models on this specific dummy dataset.
*   ResNet101's accuracy and loss are closer to DenseNet201 than ResNet50, suggesting a benefit from its deeper architecture over ResNet50, though still not surpassing DenseNet201 in this limited test.
*   ResNet50 trained the fastest, followed by ResNet101, and then DenseNet201, which was the slowest. This is generally expected as model complexity (and thus training time) often increases with depth/parameters.

**Important Note:**

These results are based on training with a small, randomly generated dummy dataset for only 5 epochs. In a real-world scenario with a proper dataset and sufficient training, the performance and relative differences between these architectures would likely vary significantly.

---
### Implementing ResNet101 with Frozen Backbone

Let's extend our comparison by implementing ResNet101. We'll follow the same methodology: ImageNet pre-trained weights, freezing the backbone, and training a new classifier head using the same dummy data.

In [14]:
from tensorflow.keras.applications import ResNet101

# Load ResNet101 with pre-trained ImageNet weights, excluding the top classifier
base_model_resnet101 = ResNet101(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the layers of the base model
for layer in base_model_resnet101.layers:
    layer.trainable = False

print("ResNet101 base model loaded and layers frozen.")

171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
ResNet101 base model loaded and layers frozen.


### Build a new classifier head for ResNet101

We'll attach a new classifier head to the frozen ResNet101 backbone, consistent with the previous models.

In [15]:
# Create a new classifier head for ResNet101
x_resnet101 = base_model_resnet101.output
x_resnet101 = GlobalAveragePooling2D()(x_resnet101)
x_resnet101 = Dense(1024, activation='relu')(x_resnet101)
predictions_resnet101 = Dense(num_classes, activation='softmax')(x_resnet101) # Use num_classes from previous dummy data setup

# Combine the base model and the new head into a full model
model_resnet101 = Model(inputs=base_model_resnet101.input, outputs=predictions_resnet101)

model_resnet101.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_2[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 44,766,602 (170.77 MB)

 Trainable params: 2,108,426 (8.04 MB)

 Non-trainable params: 42,658,176 (162.73 MB)

### Compile the ResNet101 model

In [16]:
# Compile the ResNet101 model
model_resnet101.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("ResNet101 model compiled successfully.")

ResNet101 model compiled successfully.


### Train the ResNet101 Model

Training the ResNet101 model using the same dummy data and capturing metrics.

In [17]:
import time

# Reuse epochs and batch_size from previous setup

start_time_resnet101 = time.time()
history_resnet101 = model_resnet101.fit(
    X_train_dummy, # Use the previously generated dummy data
    y_train_dummy, # Use the previously generated dummy labels
    epochs=epochs,
    batch_size=batch_size,
    verbose=1
)
end_time_resnet101 = time.time()

training_time_resnet101 = end_time_resnet101 - start_time_resnet101

print(f"\nResNet101 Training complete in {training_time_resnet101:.2f} seconds.")

Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 25s 2s/step - accuracy: 0.1000 - loss: 2.6717
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 136ms/step - accuracy: 0.1600 - loss: 2.4397
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 136ms/step - accuracy: 0.1000 - loss: 2.5861
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 132ms/step - accuracy: 0.1000 - loss: 2.4150
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step - accuracy: 0.1200 - loss: 2.3983

ResNet101 Training complete in 28.72 seconds.


### Report Metrics for ResNet101: Accuracy, Loss, and Training Time

In [18]:
# Extract metrics from history_resnet101
final_accuracy_resnet101 = history_resnet101.history['accuracy'][-1]
final_loss_resnet101 = history_resnet101.history['loss'][-1]

print(f"ResNet101 Final Training Accuracy: {final_accuracy_resnet101:.4f}")
print(f"ResNet101 Final Training Loss: {final_loss_resnet101:.4f}")
print(f"ResNet101 Total Training Time: {training_time_resnet101:.2f} seconds")

ResNet101 Final Training Accuracy: 0.1200
ResNet101 Final Training Loss: 2.3983
ResNet101 Total Training Time: 28.72 seconds


---
### Generating Dummy Test Data

To calculate additional metrics like Precision, Recall, and F1-Score, we need a separate test dataset. I'll generate dummy test data similar to the training data.

In [19]:
# Generate dummy test data
num_test_samples = 20 # A smaller test set for demonstration
X_test_dummy = np.random.rand(num_test_samples, 224, 224, 3).astype('float32')
y_test_dummy = tf.keras.utils.to_categorical(np.random.randint(0, num_classes, num_test_samples), num_classes=num_classes)

print(f"Dummy test data shape: {X_test_dummy.shape}")
print(f"Dummy test labels shape: {y_test_dummy.shape}")

Dummy test data shape: (20, 224, 224, 3)
Dummy test labels shape: (20, 10)


### Evaluating Models and Calculating Additional Metrics

Now, I'll evaluate each trained model (DenseNet201, ResNet50, ResNet101) on the newly generated dummy test data to obtain metrics like Precision, Recall, and F1-Score. I'll also collect model parameter counts as an indicator of computational cost.

In [20]:
from sklearn.metrics import precision_recall_fscore_support

def evaluate_model_metrics(model, X_test, y_test, model_name):
    print(f"\nEvaluating {model_name}...")

    # Evaluate overall accuracy and loss
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

    # Get predictions for precision, recall, f1-score
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)

    # Calculate precision, recall, f1-score (using average='weighted' for multi-class)
    precision, recall, f1_score, _ = precision_recall_fscore_support(y_true_classes, y_pred_classes, average='weighted', zero_division=0)

    # Get parameter counts
    trainable_params = sum([K.count_params(w) for w in model.trainable_weights])
    non_trainable_params = sum([K.count_params(w) for w in model.non_trainable_weights])
    total_params = trainable_params + non_trainable_params

    return {
        'accuracy': accuracy,
        'loss': loss,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'trainable_params': trainable_params,
        'non_trainable_params': non_trainable_params,
        'total_params': total_params
    }

# Need to import K from tensorflow.keras backend for parameter counting
from tensorflow.keras import backend as K

# Evaluate DenseNet201
densenet_metrics = evaluate_model_metrics(model, X_test_dummy, y_test_dummy, "DenseNet201")
densenet_metrics['training_time'] = training_time

# Evaluate ResNet50
resnet50_metrics = evaluate_model_metrics(model_resnet, X_test_dummy, y_test_dummy, "ResNet50")
resnet50_metrics['training_time'] = training_time_resnet

# Evaluate ResNet101
resnet101_metrics = evaluate_model_metrics(model_resnet101, X_test_dummy, y_test_dummy, "ResNet101")
resnet101_metrics['training_time'] = training_time_resnet101

print("\nAll models evaluated.")


Evaluating DenseNet201...
1/1 ━━━━━━━━━━━━━━━━━━━━ 24s 24s/step

Evaluating ResNet50...
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step

Evaluating ResNet101...
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step

All models evaluated.


---
### Comprehensive Performance Comparison

Here's an updated comparison table including Training Accuracy, Total Parameters, Training Time, and Test (Validation) Accuracy for DenseNet201, ResNet50, and ResNet101:

| Model             | Training Accuracy | Total Parameters | Training Time (seconds) | Test Accuracy |
| :---------------- | :---------------- | :--------------- | :---------------------- | :------------ |
| DenseNet201       | {:.4f}            | {:,}             | {:.2f}                  | {:.4f}        |
| ResNet50          | {:.4f}            | {:,}             | {:.2f}                  | {:.4f}        |
| ResNet101         | {:.4f}            | {:,}             | {:.2f}                  | {:.4f}        |

**Observations:**

*   **Accuracy:** DenseNet201 achieved the highest training and test accuracy on this specific dummy dataset.
*   **Parameters:** ResNet101 has the highest total number of parameters, making it the most complex model, followed by ResNet50, and then DenseNet201.
*   **Training Time:** ResNet50 was the fastest to train, while DenseNet201 took the longest.

**Important Note:**

These results are based on a small, randomly generated dummy dataset and limited training. In a real-world scenario with a proper dataset and sufficient training, the performance and relative differences between these architectures would likely vary significantly.

In [22]:
print(f"""
| Model             | Training Accuracy | Total Parameters | Training Time (seconds) | Test Accuracy |
| :---------------- | :---------------- | :--------------- | :---------------------- | :------------ |
| DenseNet201       | {final_accuracy:.4f}            | {densenet_metrics['total_params']:,}             | {training_time:.2f}                  | {densenet_metrics['accuracy']:.4f}        |
| ResNet50          | {final_accuracy_resnet:.4f}            | {resnet50_metrics['total_params']:,}             | {training_time_resnet:.2f}                  | {resnet50_metrics['accuracy']:.4f}        |
| ResNet101         | {final_accuracy_resnet101:.4f}            | {resnet101_metrics['total_params']:,}             | {training_time_resnet101:.2f}                  | {resnet101_metrics['accuracy']:.4f}        |
""")


| Model             | Training Accuracy | Total Parameters | Training Time (seconds) | Test Accuracy |
| :---------------- | :---------------- | :--------------- | :---------------------- | :------------ |
| DenseNet201       | 0.1300            | 20,299,338             | 77.02                  | 0.1500        |
| ResNet50          | 0.0400            | 25,696,138             | 18.69                  | 0.0500        |
| ResNet101         | 0.1200            | 44,766,602             | 28.72                  | 0.0500        |



---
### Fine-tuning DenseNet201

Based on our initial comparisons, DenseNet201 showed the best performance. Now, let's try to fine-tune it by unfreezing the last few layers of its backbone and continuing training. This often helps the model adapt more specifically to our dataset.

First, we'll create a copy of the original DenseNet201 model so we don't overwrite our previous results.

In [23]:
from tensorflow.keras.models import clone_model

# Create a copy of the original DenseNet201 model
model_finetune = clone_model(model)
model_finetune.set_weights(model.get_weights())

print("DenseNet201 model cloned for fine-tuning.")

DenseNet201 model cloned for fine-tuning.


### Unfreeze the last layers of DenseNet201 for fine-tuning

We'll unfreeze a portion of the `base_model` (DenseNet201 backbone) to allow for fine-tuning. Typically, the top layers are more task-specific, so unfreezing them can improve performance.

In [24]:
# Determine how many layers to unfreeze. Let's unfreeze the last 20 layers of the DenseNet201 base model.
# The DenseNet201 base model has many layers, so we need to be careful not to unfreeze too many or too few.

# First, check the number of layers in the base model
num_base_layers = len(base_model.layers)
print(f"Total layers in DenseNet201 base model: {num_base_layers}")

# Unfreeze the last 'n' layers of the base model within the cloned model
# Find the corresponding layers in `model_finetune.layers` that belong to `base_model_finetune`

# Re-access the base model within model_finetune structure, or simply iterate through its layers.
# Assuming 'base_model_finetune' is the embedded base model within 'model_finetune'
# A more robust way is to iterate layers of the `model_finetune` and check if they are part of the original base_model's type

# A simpler approach for demonstration: assume base_model_finetune is the first non-input layer that's frozen
# Since `clone_model` copies weights and structure, we need to re-identify the base model within it.

# A better way is to iterate through the base model's layers (from the original `base_model` object)
# and apply trainable=True to the corresponding layers in the `model_finetune` if they are in the last 'n' layers.

# For simplicity here, let's assume `model_finetune.layers` directly contains the layers we need to modify
# and that the original `base_model` structure is preserved. We'll unfreeze the last 20 layers of the entire model_finetune.

# However, the correct way is to only unfreeze layers of the *original* base_model within the new combined model.

# Let's adjust the original `base_model` variable's layers directly within the context of the `model_finetune`.
# We need to target layers of the underlying DenseNet201 within `model_finetune`.

# The structure of `model_finetune` is: Input -> DenseNet201_backbone -> GAP -> Dense -> Softmax
# We need to unfreeze layers within the DenseNet201_backbone part.

# Let's find the base model's layers in the cloned model
# For `Model` subclass, `model.layers` gives layers in definition order.
# For functional API, the base model is usually directly one of the earlier layers (if directly composed)
# A common pattern is to unfreeze `base_model` layers specifically.

# To unfreeze layers in `base_model_finetune` part of `model_finetune`

# The current setup makes `base_model` a global variable from previous code.
# We need to explicitly iterate and set `trainable=True` for the last N layers of the *cloned* base model within `model_finetune`.

# A safer approach for unfreezing layers: Iterate through the base_model (the original one) and apply to the cloned model.

for layer in model_finetune.layers[-20:]:
    if not isinstance(layer, (Dense, GlobalAveragePooling2D)): # Avoid unfreezing our newly added head if it falls in the last 20
        layer.trainable = True
        print(f"Unfreezing layer: {layer.name}")

# Important: Recompile the model after unfreezing layers
# It's common to use a lower learning rate for fine-tuning
model_finetune.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

print("DenseNet201 model recompiled with unfrozen layers and lower learning rate.")
model_finetune.summary()

Total layers in DenseNet201 base model: 707
Unfreezing layer: conv5_block30_concat
Unfreezing layer: conv5_block31_0_bn
Unfreezing layer: conv5_block31_0_relu
Unfreezing layer: conv5_block31_1_conv
Unfreezing layer: conv5_block31_1_bn
Unfreezing layer: conv5_block31_1_relu
Unfreezing layer: conv5_block31_2_conv
Unfreezing layer: conv5_block31_concat
Unfreezing layer: conv5_block32_0_bn
Unfreezing layer: conv5_block32_0_relu
Unfreezing layer: conv5_block32_1_conv
Unfreezing layer: conv5_block32_1_bn
Unfreezing layer: conv5_block32_1_relu
Unfreezing layer: conv5_block32_2_conv
Unfreezing layer: conv5_block32_concat
Unfreezing layer: bn
Unfreezing layer: relu
DenseNet201 model recompiled with unfrozen layers and lower learning rate.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ keras_tensor        │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ keras_tensor[0][… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 20,299,338 (77.44 MB)

 Trainable params: 2,542,154 (9.70 MB)

 Non-trainable params: 17,757,184 (67.74 MB)

### Fine-tune the DenseNet201 Model

Now, let's continue training the fine-tuned DenseNet201 model for additional epochs.

In [25]:
start_time_finetune = time.time()
history_finetune = model_finetune.fit(
    X_train_dummy, # Use the previously generated dummy data
    y_train_dummy, # Use the previously generated dummy labels
    epochs=epochs, # Continue for the same number of epochs, or more if desired
    batch_size=batch_size,
    verbose=1
)
end_time_finetune = time.time()

training_time_finetune = end_time_finetune - start_time_finetune

print(f"\nDenseNet201 Fine-tuning complete in {training_time_finetune:.2f} seconds.")

Epoch 1/5


4/4 ━━━━━━━━━━━━━━━━━━━━ 80s 10s/step - accuracy: 0.2500 - loss: 2.1070
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 0.2700 - loss: 2.0521
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 113ms/step - accuracy: 0.3000 - loss: 2.0099
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - accuracy: 0.3400 - loss: 1.9786
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 113ms/step - accuracy: 0.3800 - loss: 1.9548

DenseNet201 Fine-tuning complete in 82.65 seconds.


In [27]:
import os

# Define the path in Google Drive where the model will be saved
save_path = '/content/drive/MyDrive/fine_tuned_densenet201_model.keras'

# Save the fine-tuned model
model_finetune.save(save_path)

print(f"Fine-tuned DenseNet201 model saved to: {save_path}")

Fine-tuned DenseNet201 model saved to: /content/drive/MyDrive/fine_tuned_densenet201_model.keras


### Evaluate Fine-tuned DenseNet201 and Compare Performance

We'll evaluate the fine-tuned model on the dummy test set and then present a comparison table of DenseNet201 (frozen vs. fine-tuned).

In [26]:
# Evaluate fine-tuned DenseNet201
densenet_finetune_metrics = evaluate_model_metrics(model_finetune, X_test_dummy, y_test_dummy, "DenseNet201 (Fine-tuned)")
densenet_finetune_metrics['training_time'] = training_time_finetune

# Display comparison
print(f"""
### DenseNet201 Performance Comparison (Frozen Backbone vs. Fine-tuned)

| Metric              | DenseNet201 (Frozen)  | DenseNet201 (Fine-tuned) |
| :------------------ | :-------------------- | :----------------------- |
| **Training Accuracy** | {final_accuracy:.4f}            | {history_finetune.history['accuracy'][-1]:.4f} |
| **Test Accuracy**   | {densenet_metrics['accuracy']:.4f}                | {densenet_finetune_metrics['accuracy']:.4f}        |
| **Test Loss**       | {densenet_metrics['loss']:.4f}                | {densenet_finetune_metrics['loss']:.4f}           |
| **Precision**       | {densenet_metrics['precision']:.4f}                | {densenet_finetune_metrics['precision']:.4f}      |
| **Recall**          | {densenet_metrics['recall']:.4f}                | {densenet_finetune_metrics['recall']:.4f}         |
| **F1-Score**        | {densenet_metrics['f1_score']:.4f}                | {densenet_finetune_metrics['f1_score']:.4f}       |
| **Trainable Params**| {densenet_metrics['trainable_params']:,}                  | {densenet_finetune_metrics['trainable_params']:,} |
| **Total Params**    | {densenet_metrics['total_params']:,}                  | {densenet_finetune_metrics['total_params']:,}     |
| **Training Time**   | {training_time:.2f} seconds        | {training_time_finetune:.2f} seconds       |
""")


Evaluating DenseNet201 (Fine-tuned)...
1/1 ━━━━━━━━━━━━━━━━━━━━ 25s 25s/step

### DenseNet201 Performance Comparison (Frozen Backbone vs. Fine-tuned)

| Metric              | DenseNet201 (Frozen)  | DenseNet201 (Fine-tuned) |
| :------------------ | :-------------------- | :----------------------- |
| **Training Accuracy** | 0.1300            | 0.3800 |
| **Test Accuracy**   | 0.1500                | 0.2000        |
| **Test Loss**       | 2.2604                | 2.2397           |
| **Precision**       | 0.0479                | 0.1225      |
| **Recall**          | 0.1500                | 0.2000         |
| **F1-Score**        | 0.0670                | 0.1389       |
| **Trainable Params**| 1,977,354                  | 2,542,154 |
| **Total Params**    | 20,299,338                  | 20,299,338     |
| **Training Time**   | 77.02 seconds        | 82.65 seconds       |

